# Week 7: ANPR Capstone Project

## Introduction

In this week, I will combine object detection and Optical Character Recognition (OCR) to build an Automatic Number Plate Recognition (ANPR) pipeline. The system will detect a vehicle license plate using YOLO and then use EasyOCR to recognize the text on the detected license plate.

The complete workflow will be:

Image → YOLO License Plate Detection → License Plate Crop → EasyOCR → Text Recognition

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".pt", ".py", ".jpg", ".jpeg", ".png", ".yaml", ".yml")):
            print(os.path.join(root, file))

## Step 2: Install Required Libraries

In [ ]:
!pip install -q ultralytics easyocr

print("Required libraries installed successfully!")

### Findings

The required libraries for the ANPR project were installed successfully. Ultralytics will be used for YOLO object detection, while EasyOCR will be used for license plate text recognition.

## Step 3: Import Required Libraries

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import easyocr
from ultralytics import YOLO

print("Libraries imported successfully!")

### Findings

The required Python libraries were imported successfully. These libraries will be used for deep learning, image processing, YOLO object detection, image visualization, and OCR.

## Step 4: Check GPU and PyTorch

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

### Findings

PyTorch is installed successfully with CUDA support. CUDA is available and the Tesla T4 GPU is detected, allowing GPU acceleration to be used for the ANPR pipeline.

## Step 5: Download the License Plate Dataset

In [ ]:

import os
import zipfile
import subprocess

zip_path = "/content/archive.zip"

download_url = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "sujaymann/car-number-plate-dataset-yolo-format"
)

if os.path.exists(zip_path):
    os.remove(zip_path)

download = subprocess.run(
    ["wget", "-O", zip_path, download_url],
    capture_output=True,
    text=True
)

if download.returncode != 0:
    raise RuntimeError("Dataset download failed:\n" + download.stderr)

if not os.path.isfile(zip_path):
    raise FileNotFoundError("Downloaded ZIP file does not exist.")

if os.path.getsize(zip_path) == 0:
    raise ValueError("Downloaded ZIP file is empty.")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    bad_file = zip_ref.testzip()

    if bad_file is not None:
        raise ValueError(f"Corrupted ZIP file: {bad_file}")

print("Dataset downloaded successfully!")
print("ZIP file verified successfully!")
print("File size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))


### Findings

The license plate dataset was downloaded successfully from Kaggle. The dataset contains license plate images and YOLO-format annotation files that will be used for training the object detection model.

## Step 6: Extract the License Plate Dataset

In [ ]:
import zipfile
import os

zip_path = "/content/archive.zip"
extract_path = "/content/license_plate_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("Location:", extract_path)

### Findings

The license plate dataset was extracted successfully in Google Colab. The dataset is now available locally and can be prepared for YOLO training.

## Step 7: Check Dataset Structure

In [ ]:
import os

dataset_path = "/content/license_plate_dataset"

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = "  " * level

    print(indent + os.path.basename(root) + "/")

    if level < 2:
        for file in files[:10]:
            print(indent + "  " + file)

In [ ]:
import os

dataset_path = "/content/license_plate_dataset/License-Plate-Data"

train_images = os.path.join(dataset_path, "train", "images")
train_labels = os.path.join(dataset_path, "train", "labels")

test_images = os.path.join(dataset_path, "test", "images")
test_labels = os.path.join(dataset_path, "test", "labels")

print("Training images:", len(os.listdir(train_images)))
print("Training labels:", len(os.listdir(train_labels)))

print("Testing images:", len(os.listdir(test_images)))
print("Testing labels:", len(os.listdir(test_labels)))

### Findings

The dataset structure was checked successfully. The dataset contains the training and testing folders along with the image and label files required for YOLO object detection.

## Step 8: Check Dataset Configuration

In [ ]:
yaml_path = "/content/license_plate_dataset/License-Plate-Data/data.yaml"

with open(yaml_path, "r") as file:
    yaml_content = file.read()

print(yaml_content)

### Findings

The data.yaml file was successfully loaded. It contains the dataset configuration and information about the license plate object class. The paths will be updated to the local Google Colab dataset location before YOLO training.

## Step 9: Count Training and Testing Images

In [ ]:
import os

train_images = "/content/license_plate_dataset/License-Plate-Data/train/images"
test_images = "/content/license_plate_dataset/License-Plate-Data/test/images"

train_files = [
    f for f in os.listdir(train_images)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

test_files = [
    f for f in os.listdir(test_images)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Training images:", len(train_files))
print("Testing images:", len(test_files))

### Findings

The dataset contains 346 training images and 87 testing images. These images will be used to train and evaluate the license plate detection model.

## Step 10: Update data.yaml Paths for Google Colab

In [ ]:
import os
import random
import shutil

source_path = "/content/license_plate_dataset/License-Plate-Data"
corrected_path = "/content/license_plate_dataset/License-Plate-Data-corrected"

source_train_images = os.path.join(source_path, "train", "images")
source_train_labels = os.path.join(source_path, "train", "labels")

source_test_images = os.path.join(source_path, "test", "images")
source_test_labels = os.path.join(source_path, "test", "labels")

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

image_files = sorted([
    f for f in os.listdir(source_train_images)
    if f.lower().endswith(image_extensions)
])


for image in image_files:
    label = os.path.splitext(image)[0] + ".txt"
    assert os.path.isfile(os.path.join(source_train_labels, label)), \
        f"Missing label: {label}"

assert len(image_files) == 346


random.Random(42).shuffle(image_files)

train_files = image_files[:277]
val_files = image_files[277:]


test_files = sorted([
    f for f in os.listdir(source_test_images)
    if f.lower().endswith(image_extensions)
])

assert len(test_files) == 87

for image in test_files:
    label = os.path.splitext(image)[0] + ".txt"
    assert os.path.isfile(os.path.join(source_test_labels, label)), \
        f"Missing test label: {label}"


if os.path.exists(corrected_path):
    shutil.rmtree(corrected_path)

for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(corrected_path, split, "images"))
    os.makedirs(os.path.join(corrected_path, split, "labels"))

def copy_split(files, source_images, source_labels, destination):
    for filename in files:
        label = os.path.splitext(filename)[0] + ".txt"

        shutil.copy2(
            os.path.join(source_images, filename),
            os.path.join(destination, "images", filename)
        )

        shutil.copy2(
            os.path.join(source_labels, label),
            os.path.join(destination, "labels", label)
        )

copy_split(
    train_files, source_train_images, source_train_labels,
    os.path.join(corrected_path, "train")
)

copy_split(
    val_files, source_train_images, source_train_labels,
    os.path.join(corrected_path, "valid")
)

copy_split(
    test_files, source_test_images, source_test_labels,
    os.path.join(corrected_path, "test")
)

print("Corrected dataset created successfully!")
print("Training images:", len(train_files))
print("Validation images:", len(val_files))
print("Testing images:", len(test_files))

In [ ]:
import os

base_path = corrected_path

def get_stems(folder):
    return {
        os.path.splitext(f)[0]
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    }

train = get_stems(os.path.join(base_path, "train", "images"))
valid = get_stems(os.path.join(base_path, "valid", "images"))
test = get_stems(os.path.join(base_path, "test", "images"))

print("Training images:", len(train))
print("Validation images:", len(valid))
print("Test images:", len(test))

print("Training-validation overlap:", len(train & valid))
print("Validation-test overlap:", len(valid & test))
print("Training-test overlap:", len(train & test))

In [ ]:
import os

corrected_yaml_path = os.path.join(corrected_path, "data.yaml")

yaml_content = f"""path: {corrected_path}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: license_plate
"""

with open(corrected_yaml_path, "w") as file:
    file.write(yaml_content)

print("Corrected data.yaml created successfully!")
print("\nUpdated configuration:")

with open(corrected_yaml_path, "r") as file:
    print(file.read())

### Findings

The data.yaml file was updated successfully with the local Google Colab paths. The dataset is now correctly configured for YOLO training, with one object class named license_plate.

## Step 11: Explore a Sample Training Image

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

train_images_path = "/content/license_plate_dataset/License-Plate-Data/train/images"
train_labels_path = "/content/license_plate_dataset/License-Plate-Data/train/labels"


image_files = [
    f for f in os.listdir(train_images_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

image_name = image_files[0]

image_path = os.path.join(train_images_path, image_name)
label_path = os.path.join(
    train_labels_path,
    os.path.splitext(image_name)[0] + ".txt"
)


image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

height, width = image.shape[:2]


with open(label_path, "r") as file:
    labels = file.readlines()

for label in labels:
    class_id, x_center, y_center, box_width, box_height = map(
        float, label.strip().split()
    )

    x_center *= width
    y_center *= height
    box_width *= width
    box_height *= height

    x1 = int(x_center - box_width / 2)
    y1 = int(y_center - box_height / 2)
    x2 = int(x_center + box_width / 2)
    y2 = int(y_center + box_height / 2)

    cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)

plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title("Sample Training Image with License Plate Bounding Box")
plt.axis("off")
plt.show()

print("Image:", image_name)
print("Image size:", width, "x", height)
print("Number of annotations:", len(labels))

### Findings

A sample training image was successfully loaded from the dataset. The corresponding YOLO annotation was also loaded and displayed as a bounding box around the license plate. This confirms that the images and annotations are available for object detection.

## Step 12: Initialize the Pretrained YOLO Model

In [ ]:

model = YOLO("yolo26n.pt")

print("Pretrained YOLO model loaded successfully!")

### Findings

A pretrained YOLO26n model was loaded successfully. The pretrained model will be fine-tuned on the license plate dataset using transfer learning.

> **Note:** The following is the initial training run. Its validation set overlapped with the test set, so its metrics are preliminary and are not used as the final evaluation results. A corrected training run with a separate validation set is documented later in this notebook.

## Step 13: Fine-Tune the YOLO Model

In [ ]:
results = model.train(
    data=corrected_yaml_path,
    epochs=20,
    imgsz=640,
    batch=16,
    device=0
)

print("YOLO model training completed!")
print("Training results saved to:", results.save_dir)

### Findings

The YOLO26n model was successfully fine-tuned on 346 training images for 20 epochs using the Tesla T4 GPU. The best model achieved a precision of 0.895, recall of 0.883, mAP@50 of 0.930, and mAP@50-95 of 0.565 on the 87-image validation set. The trained model was saved in the training results directory.

## Step 14: Load the Best Trained YOLO Model

In [ ]:
best_model_path = os.path.join(
    str(results.save_dir),
    "weights",
    "best.pt"
)

best_model = YOLO(best_model_path)

print("Best trained YOLO model loaded successfully!")
print("Model path:", best_model_path)
print("Model exists:", os.path.exists(best_model_path))

### Findings

The best trained YOLO26n model was loaded successfully from the training results directory. This trained model will be used for license plate detection on unseen images.

## Step 15: Select an Unseen Test Image

In [ ]:
import os

test_images_path = "/content/license_plate_dataset/License-Plate-Data/test/images"

test_image_name = "Cars164.png"
test_image_path = os.path.join(test_images_path, test_image_name)

if not os.path.exists(test_image_path):
    raise FileNotFoundError(f"Test image not found: {test_image_path}")

print("Selected test image:", test_image_name)
print("Image path:", test_image_path)
print("Image exists:", os.path.exists(test_image_path))

### Findings

An unseen test image named `Cars164.png` was selected successfully from the test dataset. The test dataset contains 87 images, and this image was used to evaluate the trained YOLO model.

## Step 16: Detect the License Plate Using YOLO

In [ ]:
detection_results = best_model.predict(
    source=test_image_path,
    conf=0.25,
    device=0
)

print("YOLO detection completed successfully!")
print("Number of detected objects:", len(detection_results[0].boxes))

### Findings

The trained YOLO26n model successfully detected one license plate in the unseen test image. The detection was performed using the Tesla T4 GPU, with an inference time of approximately 9.6 ms.

## Step 17: Display the Detected License Plate

In [ ]:
annotated_image = detection_results[0].plot()

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
plt.title("YOLO License Plate Detection")
plt.axis("off")
plt.show()

### Findings

The selected test image was displayed with the YOLO detection result. The detected license plate was successfully highlighted by a bounding box.

## Step 27: Verify the Saved ANPR Result

In [ ]:
output_file = "anpr_result.txt"

with open(output_file, "r") as file:
    saved_result = file.read()

print(saved_result)

### Findings

The saved ANPR result was successfully read and verified. The saved information matches the final ANPR summary, including the input image, YOLO confidence, bounding box, OCR text, and OCR confidence.

##Inspect Available Project Files

In [ ]:
import os

print("Files available in this Colab:")

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".pt", ".py", ".ipynb", ".jpg", ".jpeg", ".png", ".yaml", ".yml")):
            print(os.path.join(root, file))

In [ ]:
!wget -O /content/archive.zip "https://www.kaggle.com/api/v1/datasets/download/sujaymann/car-number-plate-dataset-yolo-format"

In [ ]:
import zipfile
import os

zip_path = "/content/archive.zip"
extract_path = "/content/license_plate_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("Folder exists:", os.path.exists(extract_path))
print("Contents:", os.listdir(extract_path)[:20])

In [ ]:
yaml_path = "/content/license_plate_dataset/License-Plate-Data/data.yaml"

with open(yaml_path, "r") as file:
    print(file.read())

In [ ]:
import os

base_path = "/content/license_plate_dataset/License-Plate-Data"

for split in ["train", "test"]:
    image_dir = os.path.join(base_path, split, "images")
    label_dir = os.path.join(base_path, split, "labels")

    images = {
        os.path.splitext(f)[0]
        for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    }

    labels = {
        os.path.splitext(f)[0]
        for f in os.listdir(label_dir)
        if f.lower().endswith(".txt")
    }

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Images without matching labels:", len(images - labels))
    print("Labels without matching images:", len(labels - images))

In [ ]:
import os
import random
import shutil

base_path = "/content/license_plate_dataset/License-Plate-Data"

train_images = os.path.join(base_path, "train", "images")
train_labels = os.path.join(base_path, "train", "labels")

valid_images = os.path.join(base_path, "valid", "images")
valid_labels = os.path.join(base_path, "valid", "labels")

if os.path.exists(os.path.join(base_path, "valid")):
    print("A validation folder already exists. No files were copied.")
else:
    image_files = sorted([
        f for f in os.listdir(train_images)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ])

    random.seed(42)
    random.shuffle(image_files)

    valid_count = round(len(image_files) * 0.20)
    valid_files = image_files[:valid_count]

    os.makedirs(valid_images, exist_ok=True)
    os.makedirs(valid_labels, exist_ok=True)

    for image_file in valid_files:
        stem = os.path.splitext(image_file)[0]
        label_file = stem + ".txt"

        if not os.path.isfile(os.path.join(train_labels, label_file)):
            raise FileNotFoundError(f"Missing label: {label_file}")

    for image_file in valid_files:
        stem = os.path.splitext(image_file)[0]
        label_file = stem + ".txt"

        shutil.copy2(
            os.path.join(train_images, image_file),
            os.path.join(valid_images, image_file)
        )
        shutil.copy2(
            os.path.join(train_labels, label_file),
            os.path.join(valid_labels, label_file)
        )

    print("Validation split created successfully.")
    print("Validation images:", len(os.listdir(valid_images)))
    print("Validation labels:", len(os.listdir(valid_labels)))
    print("Original training images retained:", len(os.listdir(train_images)))

Validation Set Preparation

In [ ]:
import os

base_path = "/content/license_plate_dataset/License-Plate-Data"

train_dir = os.path.join(base_path, "train", "images")
valid_dir = os.path.join(base_path, "valid", "images")
test_dir = os.path.join(base_path, "test", "images")

def image_stems(folder):
    return {
        os.path.splitext(f)[0]
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    }

train_files = image_stems(train_dir)
valid_files = image_stems(valid_dir)
test_files = image_stems(test_dir)

print("Training images:", len(train_files))
print("Validation images:", len(valid_files))
print("Test images:", len(test_files))
print("Training-validation overlap:", len(train_files & valid_files))
print("Validation-test overlap:", len(valid_files & test_files))

### Findings

The initial overlap check found that all 69 validation images were also present in the training folder because the validation set was created by copying files. No overlap was found between the validation and test sets. The training file list must be adjusted so that the validation images are excluded from training. The original test set will remain unchanged for final evaluation.

In [ ]:
import os
import shutil

base_path = "/content/license_plate_dataset/License-Plate-Data"

source_images = os.path.join(base_path, "train", "images")
source_labels = os.path.join(base_path, "train", "labels")
valid_images = os.path.join(base_path, "valid", "images")

new_train_images = os.path.join(base_path, "train_corrected", "images")
new_train_labels = os.path.join(base_path, "train_corrected", "labels")

valid_stems = {
    os.path.splitext(f)[0]
    for f in os.listdir(valid_images)
}

if os.path.exists(os.path.join(base_path, "train_corrected")):
    print("train_corrected already exists. No files copied.")
else:
    os.makedirs(new_train_images, exist_ok=True)
    os.makedirs(new_train_labels, exist_ok=True)

    copied = 0

    for image_file in os.listdir(source_images):
        stem = os.path.splitext(image_file)[0]

        if stem in valid_stems:
            continue

        label_file = stem + ".txt"

        if not os.path.isfile(os.path.join(source_labels, label_file)):
            raise FileNotFoundError(f"Missing label: {label_file}")

        shutil.copy2(
            os.path.join(source_images, image_file),
            os.path.join(new_train_images, image_file)
        )
        shutil.copy2(
            os.path.join(source_labels, label_file),
            os.path.join(new_train_labels, label_file)
        )
        copied += 1

    print("Corrected training split created.")
    print("Training images copied:", copied)
    print("Validation images retained:", len(valid_stems))

### Findings

The corrected training split contains 277 images with matching label files, while the validation set contains 69 images with matching label files. The validation images have been excluded from the corrected training folder to prevent overlap between the training and validation sets. The original dataset files have been preserved, and the 87-image test set remains separate for final evaluation.

## Corrected YOLO Training with Separate Validation Data

In [ ]:
import os

base_path = "/content/license_plate_dataset/License-Plate-Data"

def get_stems(folder):
    return {

        os.path.splitext(f)[0]
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    }

train = get_stems(os.path.join(base_path, "train_corrected", "images"))
valid = get_stems(os.path.join(base_path, "valid", "images"))
test = get_stems(os.path.join(base_path, "test", "images"))

print("Corrected training images:", len(train))
print("Validation images:", len(valid))
print("Test images:", len(test))
print("Training-validation overlap:", len(train & valid))
print("Validation-test overlap:", len(valid & test))

## Updating Dataset Configuration

The dataset configuration was updated to use the correct Colab paths and a separate validation set.

In [ ]:
import os
import shutil
import yaml

base_path = "/content/license_plate_dataset/License-Plate-Data"
yaml_path = os.path.join(base_path, "data.yaml")
backup_path = os.path.join(base_path, "data_original_backup.yaml")

if not os.path.isfile(yaml_path):
    raise FileNotFoundError(f"Cannot find: {yaml_path}")

if not os.path.exists(backup_path):
    shutil.copy2(yaml_path, backup_path)

config = {
    "path": base_path,
    "train": "train_corrected/images",
    "val": "valid/images",
    "names": {0: "license_plate"},
    "nc": 1
}

with open(yaml_path, "w") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print("Updated data.yaml:")
with open(yaml_path, "r") as file:
    print(file.read())

### Findings

The dataset configuration now points to the corrected training set and separate validation set. The test set remains separate for final evaluation.

In [ ]:
%pip install ultralytics

### Findings

The Ultralytics library was installed successfully, restoring the dependency required to initialize the YOLO model.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

print("YOLO model initialized successfully!")

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training will use the CPU.")

## Verifying Dataset Configuration

Verify that YOLO uses the corrected training and validation folders while keeping the test set separate.

In [ ]:
import os
import yaml

yaml_path = "/content/license_plate_dataset/License-Plate-Data/data.yaml"

with open(yaml_path, "r") as file:
    config = yaml.safe_load(file)

print("Dataset configuration:")
print(config)

train_path = os.path.join(config["path"], config["train"])
val_path = os.path.join(config["path"], config["val"])

print("\nTraining folder exists:", os.path.isdir(train_path))
print("Validation folder exists:", os.path.isdir(val_path))
print("Test folder exists:", os.path.isdir(
    "/content/license_plate_dataset/License-Plate-Data/test"
))

### Findings

The training, validation, and test folders were verified successfully. The test set remains separate for final evaluation.

## Corrected YOLO Training with Separate Validation Data

Train YOLO for 20 epochs using the corrected dataset configuration and the available Tesla T4 GPU.

In [ ]:
results = model.train(
    data=corrected_yaml_path,
    epochs=20,
    imgsz=640,
    batch=16,
    device=0
)

print("YOLO model training completed!")
print("Training results saved to:", results.save_dir)

### Findings

YOLO training completed successfully for 20 epochs. The best model achieved 90.4% mAP@50 and 55.4% mAP@50–95 on the 69-image validation set. The separate test set has not yet been evaluated.

## Preparing the Test Set

Verify the available test images before evaluating the license plate detection and OCR pipeline.

In [ ]:
import os
from glob import glob

test_images_path = "/content/license_plate_dataset/License-Plate-Data/test/images"
test_labels_path = "/content/license_plate_dataset/License-Plate-Data/test/labels"

image_files = sorted(
    file for pattern in ("*.jpg", "*.jpeg", "*.png", "*.bmp")
    for file in glob(os.path.join(test_images_path, pattern))
)

label_files = glob(os.path.join(test_labels_path, "*.txt"))

print("Test images available:", len(image_files))
print("Test label files available:", len(label_files))
print("First 5 test images:")
for image_file in image_files[:5]:
    print(os.path.basename(image_file))

### Findings

The test set contains 87 images and 87 label files. Twenty images will be selected for final evaluation.

## Selecting 20 Test Images

Select 20 test images with their corresponding labels for evaluation of the license plate detection and OCR pipeline.

In [ ]:
import os

selected_image_names = [
    "Cars12.png", "Cars120.png", "Cars162.png", "Cars165.png",
    "Cars166.png", "Cars179.png", "Cars210.png", "Cars229.png",
    "Cars230.png", "Cars242.png", "Cars25.png", "Cars257.png",
    "Cars334.png", "Cars394.png", "Cars419.png", "Cars422.png",
    "Cars63.png", "Cars71.png", "Cars8.png", "Cars87.png"
]

image_lookup = {
    os.path.basename(path): path
    for path in image_files
}

selected_pairs = []

for image_name in selected_image_names:
    if image_name not in image_lookup:
        raise FileNotFoundError(f"Missing test image: {image_name}")

    label_name = os.path.splitext(image_name)[0] + ".txt"
    label_file = os.path.join(test_labels_path, label_name)

    if not os.path.isfile(label_file):
        raise FileNotFoundError(f"Missing label file: {label_file}")

    selected_pairs.append((image_lookup[image_name], label_file))

print("Selected test images:", len(selected_pairs))

for image_file, label_file in selected_pairs:
    print(os.path.basename(image_file), "|", os.path.basename(label_file))

### Findings

Twenty matched image-label pairs were selected from the separate test set using random seed 42. The plate text will be manually checked before evaluating OCR accuracy.

## Manually Checking Test Images

Inspect the selected test images to record the visible license plate text for comparison with the OCR predictions.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import math

for start in range(0, len(selected_pairs), 5):
    batch = selected_pairs[start:start + 5]

    fig, axes = plt.subplots(1, len(batch), figsize=(20, 5))
    if len(batch) == 1:
        axes = [axes]

    for ax, (image_file, label_file) in zip(axes, batch):
        image = Image.open(image_file).convert("RGB")
        ax.imshow(image)
        ax.set_title(os.path.basename(image_file))
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Extracting License Plates for Manual Verification

Enlarge the annotated license plate regions to make the plate text easier to inspect before evaluating OCR results.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

for image_file, label_file in selected_pairs[:5]:
    image = Image.open(image_file).convert("RGB")
    width, height = image.size

    with open(label_file, "r") as f:
        labels = f.readlines()

    print("\nImage:", os.path.basename(image_file))

    for i, line in enumerate(labels):
        values = line.split()
        if len(values) < 5:
            continue

        _, x_center, y_center, box_width, box_height = map(float, values[:5])

        x1 = int((x_center - box_width / 2) * width)
        y1 = int((y_center - box_height / 2) * height)
        x2 = int((x_center + box_width / 2) * width)
        y2 = int((y_center + box_height / 2) * height)

        crop = image.crop((
            max(0, x1), max(0, y1),
            min(width, x2), min(height, y2)
        ))

        crop = crop.resize(
            (crop.width * 4, crop.height * 4)
        )

        plt.figure(figsize=(8, 3))
        plt.imshow(crop)
        plt.title(f"{os.path.basename(image_file)} - Plate crop {i+1}")
        plt.axis("off")
        plt.show()

## Installing EasyOCR

Install EasyOCR to recognize license plate text detected by the trained YOLO model.

In [ ]:
%pip install easyocr

## Verifying EasyOCR Installation

Confirm that EasyOCR is installed and can be imported successfully.

In [ ]:
import easyocr

print("EasyOCR imported successfully!")

## Running OCR on the Selected Test Images

Apply the trained YOLO model and EasyOCR to the 20 selected test images to detect license plates and extract their text.

In [ ]:
import easyocr

reader = easyocr.Reader(["en"], gpu=True)

print("EasyOCR reader initialized successfully!")

### Findings

The YOLO and EasyOCR pipeline was tested on 20 selected test images. OCR returned text for 18 images, while no plate text was detected for 2 images. The extracted text varied in confidence and requires comparison with manually verified plate text.

## Evaluating License Plate Detection

Count successful YOLO detections on the 20 selected test images separately from OCR text recognition.

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")

detection_results = []

for image_file, label_file in selected_pairs:
    image = cv2.imread(image_file)

    predictions = model.predict(
        source=image,
        conf=0.25,
        verbose=False
    )

    detected = any(
        prediction.boxes is not None and len(prediction.boxes) > 0
        for prediction in predictions
    )

    detection_results.append({
        "image": os.path.basename(image_file),
        "plate_detected": detected
    })

detected_count = sum(
    result["plate_detected"] for result in detection_results
)

print("Test images evaluated:", len(detection_results))
print("Images with a detected plate:", detected_count)
print("Images with no detected plate:", len(detection_results) - detected_count)

for result in detection_results:
    print(result["image"], "| Plate detected:", result["plate_detected"])

### Findings

YOLO detected a license plate in 18 of the 20 test images (90% image-level detection success). No plate was detected in Cars229.png and Cars394.png. This is an image-level result, not the standard object-detection mAP metric.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

for image_file, label_file in selected_pairs[:5]:
    img = Image.open(image_file).convert("RGB")
    plt.figure(figsize=(8, 4))
    plt.imshow(img)
    plt.title(os.path.basename(image_file))
    plt.axis("off")
    plt.show()

## Manual Verification of License Plate Text

The license plate text from the 20 selected test images was manually checked and recorded for comparison with EasyOCR predictions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json

answer_key_path = "/content/drive/MyDrive/ai-internship-week8/answer_key.json"

with open(answer_key_path, "r") as file:
    answer_key_data = json.load(file)

manual_plate_text = answer_key_data["ground_truth"]
excluded_images = answer_key_data["excluded_images"]

print("Shared answer key loaded successfully!")
print("Scorable images:", len(manual_plate_text))
print("Excluded images:", len(excluded_images))

In [ ]:
print("Total scorable test images:", len(manual_plate_text))

print("\nVerified manual plate-text entries:")
for image_name, plate_text in manual_plate_text.items():
    print(f"{image_name}: {plate_text}")

blank_entries = [
    name for name, text in manual_plate_text.items()
    if not text.strip()
]

print("\nNumber of blank entries:", len(blank_entries))

if blank_entries:
    print("Images still needing entries:", blank_entries)
else:
    print("All 18 scorable entries have been filled.")

### Findings

The manual plate-text entries were checked for all 20 selected test images. Blank entries, if any, must be reviewed before calculating OCR performance.

In [ ]:
import os
import cv2
import re
from ultralytics import YOLO

best_model_path = os.path.join(
    str(results.save_dir),
    "weights",
    "best.pt"
)

model = YOLO(best_model_path)

ocr_results = []

def normalize_plate(text):
    return re.sub(r"[^A-Z0-9]", "", text.upper())

for image_file, label_file in selected_pairs:
    image = cv2.imread(image_file)

if image is None:
    raise ValueError(f"Could not read image: {image_file}")

    predictions = model.predict(
        source=image,
        conf=0.25,
        verbose=False
    )

    candidates = []

    for prediction in predictions:
        if prediction.boxes is None or len(prediction.boxes) == 0:
            continue

        for box in prediction.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)

            crop = image[
                max(0, y1):min(image.shape[0], y2),
                max(0, x1):min(image.shape[1], x2)
            ]

            if crop.size == 0:
                continue

            crop = cv2.resize(
                crop,
                None,
                fx=4,
                fy=4,
                interpolation=cv2.INTER_CUBIC
            )

            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            readings = reader.readtext(crop_rgb)

            for reading in readings:
                text = reading[1].strip()
                ocr_conf = float(reading[2])
                cleaned = normalize_plate(text)

                if 4 <= len(cleaned) <= 12:
                    candidates.append((text, ocr_conf))

    best_text, best_confidence = ("", 0.0)

    if candidates:
        best_text, best_confidence = max(
            candidates,
            key=lambda item: item[1]
        )

    ocr_results.append({
        "image": os.path.basename(image_file),
        "ocr": best_text,
        "confidence": best_confidence
    })

    print(
        f"{os.path.basename(image_file)} | "
        f"Prediction: {best_text or 'No text detected'} | "
        f"Confidence: {best_confidence:.3f}"
    )

print("\nImages evaluated:", len(ocr_results))

## Improved OCR Results: Manual Comparison

The selected OCR prediction for each test image is compared with its manually checked plate text. Images with unclear ground truth are excluded from exact-match scoring.

In [ ]:
import pandas as pd
import re

def normalize_plate(text):
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


prediction_lookup = {
    result["image"]: result["ocr"]
    for result in ocr_results
}

manual_comparison = []

for image_name, manual_text in manual_plate_text.items():

    predicted_text = prediction_lookup.get(image_name, "")

    exact_match = (
        normalize_plate(manual_text) == normalize_plate(predicted_text)
        if predicted_text else False
    )

    manual_comparison.append({
        "Image": image_name,
        "Manual Plate Text": manual_text,
        "Selected OCR Text": predicted_text or "No text detected",
        "Exact Match": exact_match,
        "Status": "Scored"
    })

comparison_df = pd.DataFrame(manual_comparison)
display(comparison_df)

total_scored = len(comparison_df)
exact_matches = int(comparison_df["Exact Match"].sum())
excluded_count = len(ocr_results) - total_scored

print("Images evaluated:", len(ocr_results))
print("Images excluded:", excluded_count)
print("Images scored:", total_scored)
print("Exact matches:", exact_matches)

print(
    "Exact-match rate:",
    f"{100 * exact_matches / total_scored:.1f}%"
    if total_scored else "N/A"
)

### Findings

The evaluation processed all 20 test images. Two images were excluded from scoring because Cars162 had a partially hidden plate and Cars87 contained two visible plates with no single clear answer. Of the remaining 18 images, the first OCR method achieved 5 exact matches (27.8%), while the whole-plate OCR method achieved 4 exact matches (22.2%). The first method performed one exact match better, but this small difference is not enough to declare a clear winner. The OCR text-recognition process requires further improvement.

## Alternative OCR Evaluation Using Paragraph Mode

EasyOCR is applied to the enlarged license plate crops using whole-line text recognition. The resulting predictions are compared with the manually checked plate text to evaluate whether reading the plate as a single line improves recognition.

In [ ]:

import os
import cv2
import pandas as pd
from ultralytics import YOLO
best_model_path = os.path.join(
    str(results.save_dir), "weights", "best.pt"
)
model = YOLO(best_model_path)

whole_plate_results = []

for image_file, label_file in selected_pairs:
    image = cv2.imread(image_file)

    best_prediction = ""

    if image is not None:
        predictions = model.predict(
            source=image,
            conf=0.25,
            verbose=False
        )

        for prediction in predictions:
            if prediction.boxes is None or len(prediction.boxes) == 0:
                continue

            for box in prediction.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)

                crop = image[
                    max(0, y1):min(image.shape[0], y2),
                    max(0, x1):min(image.shape[1], x2)
                ]

                if crop.size == 0:
                    continue

                crop = cv2.resize(
                    crop, None, fx=4, fy=4,
                    interpolation=cv2.INTER_CUBIC
                )

                crop_rgb = cv2.cvtColor(
                    crop, cv2.COLOR_BGR2RGB
                )

                readings = reader.readtext(
                    crop_rgb,
                    detail=1,
                    paragraph=True
                )

                for reading in readings:
                    text = reading[1].strip()

                    if text and len(text) > len(best_prediction):
                        best_prediction = text

    whole_plate_results.append({
        "Image": os.path.basename(image_file),
        "Whole-plate OCR prediction": best_prediction or "No text detected"
    })

whole_plate_df = pd.DataFrame(whole_plate_results)
display(whole_plate_df)

print("Images evaluated:", len(whole_plate_df))
print("Model used:", best_model_path)


### Findings

Whole-plate OCR was tested on all 20 selected images using 4× enlarged crops. Some predictions contained partial plate text or unrelated characters, and no text was detected for two images. Further refinement is needed before the OCR results can be considered reliable.

In [ ]:
from IPython.display import display
from PIL import Image
import os

check_images = [
    "Cars162.png",
    "Cars165.png",
    "Cars166.png",
    "Cars229.png",
    "Cars242.png",
    "Cars394.png",
    "Cars419.png",
    "Cars422.png"
]

for image_file, label_file in selected_pairs:
    name = os.path.basename(image_file)

    if name in check_images:
        image = Image.open(image_file).convert("RGB")
        print(name)
        display(image)

### Findings

Visual inspection showed that Cars162.png was partly blocked, Cars229.png had no clearly visible plate, and Cars242.png was blurry. The plates in Cars165.png, Cars166.png, Cars394.png, Cars419.png, and Cars422.png appeared clear. These observations help distinguish image-quality problems from OCR recognition errors.

## Review of OCR Prediction Errors

The whole-plate OCR predictions are compared with the manually checked plate text to identify mismatched characters and incomplete predictions.

In [ ]:
import re
import pandas as pd

def normalize_plate(text):
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())

whole_plate_lookup = {
    row["Image"]: row["Whole-plate OCR prediction"]
    for _, row in whole_plate_df.iterrows()
}

whole_plate_comparison = []

for image_name, manual_text in manual_plate_text.items():

    predicted_text = whole_plate_lookup.get(
        image_name, "No text detected"
    )

    exact_match = (
        normalize_plate(manual_text) == normalize_plate(predicted_text)
        if predicted_text != "No text detected"
        else False
    )

    whole_plate_comparison.append({
        "Image": image_name,
        "Manual Text": manual_text,
        "OCR Prediction": predicted_text,
        "Exact Match": exact_match,
        "Status": "Scored"
    })

whole_plate_comparison_df = pd.DataFrame(whole_plate_comparison)

display(whole_plate_comparison_df)

total_scored = len(whole_plate_comparison_df)
exact_matches = int(
    whole_plate_comparison_df["Exact Match"].sum()
)

print("Images evaluated:", len(whole_plate_df))
print("Images excluded:", len(whole_plate_df) - total_scored)
print("Images scored:", total_scored)
print("Exact matches:", exact_matches)

print(
    "Exact-match rate:",
    f"{100 * exact_matches / total_scored:.1f}%"
    if total_scored else "N/A"
)

In [ ]:
import os
from PIL import Image
from IPython.display import display

image_folder = "/content/license_plate_dataset/License-Plate-Data/test/images"

image_names = [
    "Cars12.png", "Cars120.png", "Cars162.png", "Cars165.png",
    "Cars166.png", "Cars179.png", "Cars210.png", "Cars229.png",
    "Cars230.png", "Cars242.png", "Cars25.png", "Cars257.png",
    "Cars334.png", "Cars394.png", "Cars419.png", "Cars422.png",
    "Cars63.png", "Cars71.png", "Cars8.png", "Cars87.png"
]

for name in image_names:
    path = os.path.join(image_folder, name)

    if os.path.exists(path):
        print("Image:", name)
        img = Image.open(path).convert("RGB")
        display(img.resize((1000, int(img.height * 1000 / img.width))))
        img.close()
    else:
        print("MISSING:", name)

## Findings

- YOLO was used to detect license plates, while EasyOCR was used to recognize their text.
- A total of 20 test images were evaluated.
- Two images were excluded: Cars162.png (partially hidden plate) and Cars87.png (two visible plates).
- The remaining 18 images were evaluated using the same verified answer key.
- The first OCR method achieved 5 correct predictions out of 18 (27.8%).
- The whole-plate OCR method achieved 4 correct predictions out of 18 (22.2%).
- The first method achieved only one additional correct prediction, which is not enough to establish a clear winner.

## Conclusion

The Week 7 experiment demonstrated license plate recognition using YOLO for object detection and EasyOCR for text recognition. Two OCR methods were tested using the same 18 scorable images and verified answer key.

The first OCR method achieved 27.8% exact-match accuracy, while the whole-plate OCR method achieved 22.2%. Although the first method performed slightly better, the difference was only one correct prediction.

The results show that recognizing license plate characters accurately remains challenging. Improvements in image preprocessing, plate cropping, and OCR techniques may help improve recognition performance.